In [1]:
#### install Requirements
!pip install chromadb==0.5.5 langchain-chroma==0.1.2 langchain==0.2.11 langchain-community==0.2.10 langchain-text-splitters==0.2.2 langchain-groq==0.1.6 transformers==4.43.2 sentence-transformers==3.0.1 unstructured==0.15.0 unstructured[pdf]==0.15.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... - \ | / done
  Getting requirements to build wheel ... - done
  Preparing metadata (pyproject.toml) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s et

In [2]:
#### Import Libraries
import os
import nltk
import json
# The rest of your import statements...
from langchain.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

In [3]:
### Config API
config_file_path = "/kaggle/input/api-key/config.json"
config_data = json.load(open(config_file_path))
api_key = config_data["api_key"]
os.environ["GROQ_API_KEY"] = api_key

In [4]:
# laoding the document
loader = UnstructuredFileLoader("/kaggle/input/pdf-dataset/Lecture_7-unsupervised-learning.pdf")
documents = loader.load()

In [5]:
# splitting into text chunks
text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)
texts = text_splitter.split_documents(documents)
type(texts),len(texts)

(list, 20)

In [6]:
### Load HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()

/tmp/ipykernel_23/2012617692.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
### Chroma Database
persist_directory = "doc_db"
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [8]:
# retriever
retriever = vectordb.as_retriever()

In [9]:
# llm from groq
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0
)

In [10]:
# create a qa chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [11]:
######### ASK
def ASK(query):
    response = qa_chain.invoke({"query": query})
    print(query)
    print("\n","*"*80,"\n")
    print(response["result"])
    print("\n","*"*80,"\n")
    print("Source Document:", response["source_documents"][0].metadata["source"])

In [12]:
# invoke the qa chain and get a response for user query
query = "What about this Document?"
ASK(query)

What about this Document?

 ******************************************************************************** 

This document appears to be a set of lecture slides for a data mining or machine learning course, specifically covering the topic of clustering. The slides are from a course called CS583, taught by Bing Liu at the University of Illinois at Chicago (UIC).

The slides cover various aspects of clustering, including:

1. Basic concepts: types of attributes (symmetric, asymmetric, nominal), distance functions, and clustering algorithms (K-means, hierarchical).
2. Clustering evaluation: methods for evaluating the quality of a clustering result, including external evaluation (using labeled data) and internal evaluation (using measures such as intra-cluster cohesion and inter-cluster separation).
3. Clustering applications: examples of how clustering is used in various fields, such as marketing, text document organization, and recommendation systems.
4. Clustering algorithms: K-means 

In [13]:
# invoke the qa chain and get a response for user query
query = "Can you provide summary for kmeans in this document?"
ASK(query)

Can you provide summary for kmeans in this document?

 ******************************************************************************** 

Based on the provided context, here is a summary of K-means clustering:

**K-means Clustering**

* K-means is a partitional clustering algorithm.
* It partitions the given data into k clusters, where k is specified by the user.
* Each cluster has a cluster center, called a centroid.
* The algorithm works as follows:
  1. Randomly choose k data points (seeds) to be the initial centroids.
  2. Assign each data point to the closest centroid.
  3. Re-compute the centroids using the current cluster memberships.
  4. Repeat steps 2-3 until a convergence criterion is met.

**Convergence Criterion**

* No (or minimum) re-assignments of data points to different clusters.
* No (or minimum) change of centroids.
* Minimum decrease in the sum of squared error (SSE).

**Strengths**

* Not explicitly mentioned in the provided context, but mentioned that there is a 

In [14]:
# invoke the qa chain and get a response for user query
query = "provide summary for all document?"
ASK(query)

provide summary for all document?

 ******************************************************************************** 

The document appears to be a lecture on clustering, a data mining technique. Here's a summary of the main points:

**What is Clustering?**

* Clustering is a technique used to group similar objects or data points into clusters.
* It is used in various fields such as medicine, psychology, botany, sociology, biology, archeology, marketing, insurance, and libraries.
* Clustering is used to identify patterns or structures in data that are not easily visible.

**Examples of Clustering**

* Grouping people of similar sizes together to make "small", "medium", and "large" T-Shirts.
* Segmenting customers according to their similarities for targeted marketing.
* Organizing text documents according to their content similarities.

**Aspects of Clustering**

* Clustering algorithm: Partitional clustering, Hierarchical clustering, etc.
* Distance function: Measures the similarity o

In [15]:
# invoke the qa chain and get a response for user query
query = "provide summary for clustering types?"
ASK(query)

provide summary for clustering types?

 ******************************************************************************** 

Based on the provided context, here is a summary of clustering types:

1. **Partitional Clustering**: Divides the data into a fixed number of clusters, where each cluster is a separate group of data points.
2. **Hierarchical Clustering**: Produces a nested sequence of clusters, a tree, also called a Dendrogram. It can be further divided into:
	* **Agglomerative (Bottom-up) Clustering**: Builds the dendrogram from the bottom level, merging the most similar (or nearest) pair of clusters until all data points are merged into a single cluster.
	* **Divisive (Top-down) Clustering**: Not mentioned in the provided context, but it starts with all data points in a single cluster and recursively splits the cluster into smaller ones.

Note that the provided context does not mention other types of clustering, such as **Density-based Clustering**, **Distribution-based Clusterin

In [16]:
# invoke the qa chain and get a response for user query
query = "provide summary for Hierarchical Clustering and how to apply?"
ASK(query)

provide summary for Hierarchical Clustering and how to apply?

 ******************************************************************************** 

**Hierarchical Clustering:**

Hierarchical clustering is a type of clustering algorithm that produces a nested sequence of clusters, represented as a tree-like structure called a dendrogram. This algorithm starts with each data point as a separate cluster and then merges the most similar clusters until all data points are merged into a single cluster.

**Types of Hierarchical Clustering:**

1. **Agglomerative (Bottom-up) Clustering:** This approach starts with each data point as a separate cluster and then merges the most similar clusters until all data points are merged into a single cluster.
2. **Divisive (Top-down) Clustering:** This approach starts with all data points in a single cluster and then splits the cluster into smaller clusters until each data point is in its own cluster.

**How to Apply Hierarchical Clustering:**

1. **Choose 

In [17]:
#### بس كدة لو عجبك ابدى بتعليقك